[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/25.%20The%20Dynamic%20Truck%20Appointment%20System%20Problem/P25-Tier-1_executed.ipynb)

---

# 25. The Dynamic Truck Appointment System Problem
## Tier 1: The Pen-and-Paper Method (Stochastic Programming Formulation)

### Goal

Develop and solve a two-stage stochastic mixed-integer optimization model for assigning trucks to appointment slots when arrival delays are uncertain. The first-stage decisions are made before the delay scenario is known. Scenario-specific final assignments are made after the scenario is observed.

The notebook takes as input the number of trucks and slots, slot capacity, cost parameters, and a probability distribution over delay scenarios. It returns an optimal initial assignment, a final assignment for each scenario, and a breakdown of expected costs. The model is solved as a mixed-integer linear program (MILP) using PuLP's CBC solver.

A *scenario* is one possible set of delays for all trucks. A *first-stage decision* is chosen before the scenario is known; it must therefore be the same in every scenario. A *recourse decision* is chosen after the scenario is observed and may differ between scenarios. In the model, the shared initial-assignment variables enforce this timing structure.

### Scope and modeling assumptions

- The listed scenarios and their probabilities are known, and their probabilities sum to one.
- Each truck receives exactly one initial appointment slot. Each slot has a fixed capacity measured in trucks.
- After a delay scenario is observed, each truck receives exactly one final slot. It may keep its initial slot or move to a different slot. The scenario-specific final assignment must satisfy the same slot capacity.
- The operating cost is charged for each initial truck-slot assignment. In this example, every truck receives one assignment, so this component is fixed across feasible solutions.
- Waiting cost is charged for a reassigned truck. Its waiting time is modeled as at least that truck's delay in the observed scenario. Because waiting has a positive cost, the minimizing solution sets this modeled waiting time to its lower bound.
- A rescheduling cost is charged once for each truck whose slot changes. The model does not represent slot timing, delay-dependent arrival feasibility, queueing interactions, or costs other than those explicitly defined here.
- The recourse decisions are scenario-specific: the model assumes the delay scenario is observed before those decisions are made. It does not model additional uncertainty within a scenario.

These assumptions define a compact teaching model. In particular, the delay values are used as waiting-time lower bounds for reassigned trucks; they are not used to calculate whether a truck can physically arrive before a particular slot.

### Example data

The example has three trucks and three appointment slots. Delay values are in minutes.

| Scenario | Probability | Truck 1 delay | Truck 2 delay | Truck 3 delay |
|---|---:|---:|---:|---:|
| No Delay | 0.4 | 0 | 0 | 0 |
| Light Delay | 0.4 | 15 | 20 | 10 |
| Heavy Delay | 0.2 | 45 | 60 | 30 |

Each slot can serve at most two trucks. The operating cost is $100 per initial truck-slot assignment, the waiting cost is $5 per minute, and the rescheduling cost is $50 per changed truck assignment.

### Solution sequence

1. Define the trucks, slots, scenarios, probabilities, delays, capacities, and costs.
2. Create first-stage assignment variables and scenario-specific final-assignment, change-indicator, and waiting-time variables.
3. Minimize the initial operating cost plus the probability-weighted recourse costs.
4. Enforce initial assignment, final assignment, capacity, change-indicator, and waiting-time constraints.
5. Solve the mixed-integer program, extract the assignments, and examine scenario costs and sensitivity to the rescheduling cost.

### What to examine

- The initial assignment selected before the delay scenario is known.
- How final assignments and modeled waiting times vary across scenarios.
- How operating, waiting, and rescheduling costs contribute to scenario costs and expected cost.
- How changing the rescheduling-cost parameter affects the optimum and the number of scenario-specific changes.

The model provides an optimization benchmark for this specific formulation. Comparisons with other scheduling methods require those methods and their assumptions to be defined separately.

## Mathematical model

### Sets, data, and decisions

Let $K$ be the set of trucks, $T$ the set of appointment slots, and $S$ the set of delay scenarios. For scenario $s \in S$, let $p_s$ be its probability and $d_{ks}$ the delay, in minutes, of truck $k \in K$. Let $C$ be the capacity of each slot, $c$ the operating cost per initial truck-slot assignment, $h$ the waiting cost per minute, and $r$ the cost of rescheduling one truck.

The binary variable $x_{kt}$ equals 1 when truck $k$ is initially assigned to slot $t$. The binary variable $y_{kts}$ equals 1 when truck $k$ has final slot $t$ under scenario $s$. The binary variable $q_{ks}$ equals 1 when truck $k$ changes slots under scenario $s$. The nonnegative variable $z_{kts}$ represents modeled waiting time associated with truck $k$'s final slot $t$ in scenario $s$, in minutes.

The distinction between $y$ and $q$ matters: $y$ identifies the truck's final slot, while $q$ records whether that slot differs from its initial slot. Each truck has one initial slot and one final slot. The linking constraints below make $q_{ks}$ equal to 1 exactly when those slots differ. This lets the objective charge once per changed truck, rather than once per final-slot assignment.

### Objective: minimize expected total cost

The objective combines the initial operating cost with the probability-weighted waiting and rescheduling costs in each scenario:

$$\min \quad c\sum_{k\in K}\sum_{t\in T}x_{kt} + \sum_{s\in S}p_s\left(h\sum_{k\in K}\sum_{t\in T}z_{kts} + r\sum_{k\in K}q_{ks}\right).$$

Here, the first term is paid before the scenario is known. The terms inside the sum over $s$ are scenario-specific recourse costs, weighted by the probability $p_s$. The code builds these components as `first_stage_cost` and `second_stage_cost`.

### Assignment and capacity constraints

Each truck has one initial slot, and no initial slot exceeds capacity:

$$\sum_{t\in T}x_{kt}=1 \quad \forall k\in K, \qquad \sum_{k\in K}x_{kt}\le C \quad \forall t\in T.$$

For every scenario, each truck has exactly one final slot. The final capacity constraint counts all trucks assigned to a slot in that scenario:

$$\sum_{t\in T}y_{kts}=1 \quad \forall k\in K,\ s\in S, \qquad \sum_{k\in K}y_{kts}\le C \quad \forall t\in T,\ s\in S.$$

These constraints use $y$ for the complete final assignment. A truck that keeps its initial slot is still counted in the scenario's final capacity.

### Change indicators and waiting-time constraints

The following constraints link the change indicator $q_{ks}$ to the initial and final assignments:

$$q_{ks}\ge x_{kt}-y_{kts}, \qquad q_{ks}\ge y_{kts}-x_{kt}, \qquad q_{ks}\le 2-x_{kt}-y_{kts} \quad \forall k\in K,\ t\in T,\ s\in S.$$

Because $x$ and $y$ each select exactly one slot, these inequalities force $q_{ks}=0$ when the selected initial and final slots are the same, and $q_{ks}=1$ when they differ. The upper bound is important when the rescheduling cost is zero: it makes the indicator exact even when the objective has no incentive to minimize $q$.

A reassigned truck's modeled waiting time must cover its delay at the new slot:

$$z_{kts}\ge d_{ks}(y_{kts}-x_{kt}) \quad \forall k\in K,\ t\in T,\ s\in S, \qquad z_{kts}\ge 0.$$

If a truck moves to slot $t$, then $y_{kts}=1$ and $x_{kt}=0$, so the lower bound is its delay. At the slot it leaves, the difference is negative and the nonnegativity bound applies. If it stays in its initial slot, all these lower bounds permit zero waiting. Since $h$ is positive, the minimizing solution sets each waiting variable to its smallest feasible value.

The variable domains are

$$x_{kt},y_{kts},q_{ks}\in\{0,1\}.$$

In the code, `x` stores initial assignments, `y` stores final assignments, `q` stores change indicators, and `z` stores waiting-time variables. The objective and constraints are linear, and the binary variables make this a mixed-integer linear program.

### Interpretation and computational note

The model is a two-stage stochastic mixed-integer linear program: binary first-stage decisions are shared across scenarios, while binary and continuous recourse decisions are indexed by scenario. Sharing $x$ across scenarios is the nonanticipativity condition: the initial assignment cannot depend on information that has not yet been observed. The model enumerates the supplied scenarios; it does not sample new scenarios.

The number of decision variables grows on the order of $|K||T|+|K||T||S|+|K||S|$. The mixed-integer problem can become computationally demanding as the numbers of trucks, slots, and scenarios grow. The solver returns an optimal solution for the encoded instance; if multiple optimal assignments exist, the particular one returned can depend on solver tie-breaking.

In [ ]:
from dataclasses import dataclass
from math import isclose
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import seaborn as sns

# A consistent plotting style makes the scenario and cost comparisons easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the problem data

A scenario is one possible set of truck delays, together with the probability assigned to that set. The problem data class keeps the scenario list, capacities, and cost parameters together so that the same solver can be used for the example and for the sensitivity analysis.

In [ ]:
@dataclass
class DelayScenario:
    """A delay scenario with its probability and truck-specific delays."""

    name: str
    probability: float
    delays: Dict[int, int]  # truck_id -> delay in minutes


@dataclass
class TruckAppointmentProblem:
    """Data defining a dynamic truck appointment optimization problem."""

    num_trucks: int
    num_slots: int
    slot_capacity: int
    operating_cost_per_slot: float
    waiting_cost_per_minute: float
    rescheduling_cost_per_change: float
    scenarios: List[DelayScenario]


def create_example_problem():
    """Create the three-truck, three-slot example and its delay scenarios."""
    scenarios = [
        DelayScenario("No Delay", 0.4, {1: 0, 2: 0, 3: 0}),
        DelayScenario("Light Delay", 0.4, {1: 15, 2: 20, 3: 10}),
        DelayScenario("Heavy Delay", 0.2, {1: 45, 2: 60, 3: 30}),
    ]

    return TruckAppointmentProblem(
        num_trucks=3,
        num_slots=3,
        slot_capacity=2,
        operating_cost_per_slot=100,
        waiting_cost_per_minute=5,
        rescheduling_cost_per_change=50,
        scenarios=scenarios,
    )


problem = create_example_problem()
assert isclose(
    sum(scenario.probability for scenario in problem.scenarios),
    1.0,
    rel_tol=1e-9,
    abs_tol=1e-9,
), "Scenario probabilities must sum to one."

print(
    f"Problem created: {problem.num_trucks} trucks, "
    f"{problem.num_slots} slots, {len(problem.scenarios)} scenarios"
)

## 2. Formulate and solve the stochastic program

The solver creates one shared set of initial assignments, then creates separate final-assignment, change-indicator, and waiting-time variables for every scenario. The initial assignment cannot depend on the scenario, but the recourse decisions can depend on the observed scenario. This structure represents the two decision stages directly.

In [ ]:
def solve_stochastic_programming(problem: TruckAppointmentProblem):
    """Solve the two-stage stochastic mixed-integer appointment model."""
    probability_total = sum(
        scenario.probability for scenario in problem.scenarios
    )
    if not isclose(probability_total, 1.0, rel_tol=1e-9, abs_tol=1e-9):
        raise ValueError("Scenario probabilities must sum to one.")

    expected_trucks = set(range(1, problem.num_trucks + 1))
    for scenario in problem.scenarios:
        if set(scenario.delays) != expected_trucks:
            raise ValueError(
                f"Scenario {scenario.name!r} must define a delay for every truck."
            )

    model = pulp.LpProblem(
        "Dynamic_Truck_Appointments",
        pulp.LpMinimize,
    )

    # First-stage binary decisions are shared by every scenario.
    x = {
        (k, t): pulp.LpVariable(
            f"x_{k}_{t}",
            0,
            1,
            pulp.LpBinary,
        )
        for k in range(1, problem.num_trucks + 1)
        for t in range(1, problem.num_slots + 1)
    }

    # Recourse variables distinguish final slots, changed trucks, and waiting.
    y = {}
    q = {}
    z = {}
    for s_idx, scenario in enumerate(problem.scenarios):
        for k in range(1, problem.num_trucks + 1):
            q[k, s_idx] = pulp.LpVariable(
                f"q_{k}_{s_idx}",
                0,
                1,
                pulp.LpBinary,
            )
            for t in range(1, problem.num_slots + 1):
                y[k, t, s_idx] = pulp.LpVariable(
                    f"y_{k}_{t}_{s_idx}",
                    0,
                    1,
                    pulp.LpBinary,
                )
                z[k, t, s_idx] = pulp.LpVariable(
                    f"z_{k}_{t}_{s_idx}",
                    0,
                )

    first_stage_cost = pulp.lpSum(
        problem.operating_cost_per_slot * x[k, t]
        for k in range(1, problem.num_trucks + 1)
        for t in range(1, problem.num_slots + 1)
    )

    second_stage_cost = pulp.lpSum(
        scenario.probability
        * (
            pulp.lpSum(
                problem.waiting_cost_per_minute * z[k, t, s_idx]
                for k in range(1, problem.num_trucks + 1)
                for t in range(1, problem.num_slots + 1)
            )
            + pulp.lpSum(
                problem.rescheduling_cost_per_change * q[k, s_idx]
                for k in range(1, problem.num_trucks + 1)
            )
        )
        for s_idx, scenario in enumerate(problem.scenarios)
    )

    model += first_stage_cost + second_stage_cost

    # Each truck receives exactly one initial appointment slot.
    for k in range(1, problem.num_trucks + 1):
        model += pulp.lpSum(
            x[k, t] for t in range(1, problem.num_slots + 1)
        ) == 1

    # The initial schedule must respect each slot's truck capacity.
    for t in range(1, problem.num_slots + 1):
        model += pulp.lpSum(
            x[k, t] for k in range(1, problem.num_trucks + 1)
        ) <= problem.slot_capacity

    for s_idx, scenario in enumerate(problem.scenarios):
        for k in range(1, problem.num_trucks + 1):
            # Each truck receives exactly one final slot in this scenario.
            model += pulp.lpSum(
                y[k, t, s_idx]
                for t in range(1, problem.num_slots + 1)
            ) == 1

            # These bounds make q equal one exactly when the selected slot changes.
            for t in range(1, problem.num_slots + 1):
                model += q[k, s_idx] >= x[k, t] - y[k, t, s_idx]
                model += q[k, s_idx] >= y[k, t, s_idx] - x[k, t]
                model += q[k, s_idx] <= 2 - x[k, t] - y[k, t, s_idx]

                # Waiting is incurred at the new slot, not at the slot being left.
                model += (
                    z[k, t, s_idx]
                    >= scenario.delays[k] * (y[k, t, s_idx] - x[k, t])
                )

        # Final capacity counts every truck assigned to each slot in this scenario.
        for t in range(1, problem.num_slots + 1):
            model += pulp.lpSum(
                y[k, t, s_idx]
                for k in range(1, problem.num_trucks + 1)
            ) <= problem.slot_capacity

    print("Solving the stochastic programming model...")
    model.solve(pulp.PULP_CBC_CMD(msg=0))

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(
            f"The optimization model did not reach optimality: "
            f"{pulp.LpStatus[model.status]}"
        )

    return model, x, y, z


model, x, y, z = solve_stochastic_programming(problem)
print(f"Model status: {pulp.LpStatus[model.status]}")
print(f"Objective value: ${pulp.value(model.objective):.2f}")

## 3. Extract the assignments

The optimization variables are indexed by truck, slot, and—when they are recourse variables—scenario. This function translates those indexed values into dictionaries that are easier to inspect. For every scenario, it records the final slot selected by the model and the modeled waiting time at that slot. Comparing the final and initial slots identifies whether the truck was rescheduled.

In [ ]:
def extract_solution(problem, x, y, z):
    """Extract initial and scenario-specific final assignments and waiting times."""
    initial_assignments = {}
    for k in range(1, problem.num_trucks + 1):
        selected_slots = [
            t
            for t in range(1, problem.num_slots + 1)
            if pulp.value(x[k, t]) > 0.5
        ]
        if len(selected_slots) != 1:
            raise ValueError(f"Truck {k} does not have exactly one initial slot.")
        initial_assignments[k] = selected_slots[0]

    final_assignments = {}
    waiting_times = {}

    for s_idx, _scenario in enumerate(problem.scenarios):
        final_assignments[s_idx] = {}
        waiting_times[s_idx] = {}

        for k in range(1, problem.num_trucks + 1):
            selected_slots = [
                t
                for t in range(1, problem.num_slots + 1)
                if pulp.value(y[k, t, s_idx]) > 0.5
            ]
            if len(selected_slots) != 1:
                raise ValueError(
                    f"Truck {k} does not have exactly one final slot "
                    f"in scenario {s_idx}."
                )

            slot = selected_slots[0]
            final_assignments[s_idx][k] = slot
            waiting_times[s_idx][k] = pulp.value(z[k, slot, s_idx])

    return initial_assignments, final_assignments, waiting_times


initial_assignments, final_assignments, waiting_times = extract_solution(
    problem,
    x,
    y,
    z,
)

print("Initial assignments:")
for truck, slot in initial_assignments.items():
    print(f"  Truck {truck} -> Slot {slot}")

print("\nFinal assignments by scenario:")
for s_idx, scenario in enumerate(problem.scenarios):
    print(f"\n{scenario.name} (probability={scenario.probability:.1f}):")
    for truck in range(1, problem.num_trucks + 1):
        initial_slot = initial_assignments[truck]
        final_slot = final_assignments[s_idx][truck]
        wait_time = waiting_times[s_idx][truck]
        if final_slot != initial_slot:
            print(
                f"  Truck {truck}: Slot {initial_slot} -> Slot {final_slot} "
                f"(modeled waiting: {wait_time:.0f} min; rescheduled)"
            )
        else:
            print(
                f"  Truck {truck}: Slot {final_slot} "
                f"(modeled waiting: {wait_time:.0f} min; unchanged)"
            )

## 4. Calculate scenario costs

A scenario cost is the initial operating cost plus that scenario's waiting and rescheduling costs. Expected total cost is the sum of scenario costs weighted by their probabilities. Computing the same components from the extracted assignments provides a transparent check against the objective value reported by the optimization model.

In [ ]:
def calculate_cost_breakdown(
    problem,
    initial_assignments,
    final_assignments,
    waiting_times,
):
    """Calculate operating, waiting, rescheduling, and expected costs by scenario."""
    operating_cost = (
        len(initial_assignments) * problem.operating_cost_per_slot
    )
    cost_breakdown = []

    for s_idx, scenario in enumerate(problem.scenarios):
        waiting_cost = sum(
            waiting_times[s_idx][k] * problem.waiting_cost_per_minute
            for k in range(1, problem.num_trucks + 1)
        )
        rescheduling_cost = sum(
            problem.rescheduling_cost_per_change
            for k in range(1, problem.num_trucks + 1)
            if final_assignments[s_idx][k] != initial_assignments[k]
        )
        scenario_cost = operating_cost + waiting_cost + rescheduling_cost

        cost_breakdown.append(
            {
                "Scenario": scenario.name,
                "Probability": scenario.probability,
                "Operating Cost": operating_cost,
                "Waiting Cost": waiting_cost,
                "Rescheduling Cost": rescheduling_cost,
                "Total Scenario Cost": scenario_cost,
                "Expected Cost Contribution": scenario.probability * scenario_cost,
            }
        )

    return pd.DataFrame(cost_breakdown)


cost_df = calculate_cost_breakdown(
    problem,
    initial_assignments,
    final_assignments,
    waiting_times,
)
print(cost_df.round(2).to_string(index=False))

expected_cost_from_breakdown = cost_df["Expected Cost Contribution"].sum()
assert isclose(
    expected_cost_from_breakdown,
    pulp.value(model.objective),
    rel_tol=1e-7,
    abs_tol=1e-6,
), "The scenario cost breakdown must reconcile with the model objective."

print(f"\nTotal expected cost: ${expected_cost_from_breakdown:.2f}")

## 5. Visualize assignments, costs, and waiting times

The panels show different views of the same solution: the initial assignment matrix, the cost components in each scenario, each scenario's contribution to expected cost, and the distribution of modeled waiting times across trucks. The scenario cost chart displays costs before probability weighting; the expected-cost chart displays probability-weighted contributions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Dynamic Truck Appointment System: Stochastic Programming",
    fontsize=16,
    fontweight="bold",
)

# A binary matrix makes the single initial slot assigned to each truck visible.
ax1 = axes[0, 0]
assignment_matrix = np.zeros((problem.num_trucks, problem.num_slots))
for truck, slot in initial_assignments.items():
    assignment_matrix[truck - 1, slot - 1] = 1

sns.heatmap(
    assignment_matrix,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Assignment indicator (0 or 1)"},
    ax=ax1,
    xticklabels=[f"Slot {t}" for t in range(1, problem.num_slots + 1)],
    yticklabels=[f"Truck {k}" for k in range(1, problem.num_trucks + 1)],
)
ax1.set_title("Initial Assignment Matrix")
ax1.set_xlabel("Appointment slot")
ax1.set_ylabel("Truck")

# Stacked bars separate the three modeled cost components by scenario.
ax2 = axes[0, 1]
cost_categories = ["Operating Cost", "Waiting Cost", "Rescheduling Cost"]
scenario_costs = cost_df.set_index("Scenario")[cost_categories]
scenario_costs.plot(kind="bar", stacked=True, ax=ax2)
ax2.set_title("Cost Breakdown by Scenario")
ax2.set_xlabel("Delay scenario")
ax2.set_ylabel("Scenario cost ($)")
ax2.legend(title="Cost component")
ax2.tick_params(axis="x", rotation=35)

# Probability labels explain how scenario costs contribute to expectation.
ax3 = axes[1, 0]
ax3.bar(
    cost_df["Scenario"],
    cost_df["Expected Cost Contribution"],
    color=["#1f77b4", "#ff7f0e", "#2ca02c"],
)
ax3.set_title("Expected Cost Contribution by Scenario")
ax3.set_xlabel("Delay scenario")
ax3.set_ylabel("Expected cost contribution ($)")
ax3.tick_params(axis="x", rotation=35)
for i, row in cost_df.reset_index(drop=True).iterrows():
    ax3.text(
        i,
        row["Expected Cost Contribution"],
        f"p={row['Probability']:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Each observation is one truck's modeled waiting time in one scenario.
ax4 = axes[1, 1]
waiting_data = [
    {
        "Scenario": scenario.name,
        "Truck": f"Truck {k}",
        "Waiting Time (minutes)": waiting_times[s_idx][k],
    }
    for s_idx, scenario in enumerate(problem.scenarios)
    for k in range(1, problem.num_trucks + 1)
]
waiting_df = pd.DataFrame(waiting_data)
sns.boxplot(
    data=waiting_df,
    x="Scenario",
    y="Waiting Time (minutes)",
    ax=ax4,
)
ax4.set_title("Modeled Waiting Times by Scenario")
ax4.set_xlabel("Delay scenario")
ax4.set_ylabel("Waiting time (minutes)")
ax4.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

## 6. Sensitivity to rescheduling cost

Sensitivity analysis solves the same stochastic program repeatedly while varying only the rescheduling-cost parameter. This reveals how the modeled trade-off between changing a truck's slot and waiting for its delay responds to that cost. The number of reschedules shown below is the unweighted count across all truck-scenario pairs; it is not an expected count.

In [ ]:
def sensitivity_analysis(problem):
    """Solve the model for a range of rescheduling-cost values."""
    rescheduling_costs = [0, 25, 50, 75, 100, 150]
    results = []

    for reschedule_cost in rescheduling_costs:
        modified_problem = TruckAppointmentProblem(
            num_trucks=problem.num_trucks,
            num_slots=problem.num_slots,
            slot_capacity=problem.slot_capacity,
            operating_cost_per_slot=problem.operating_cost_per_slot,
            waiting_cost_per_minute=problem.waiting_cost_per_minute,
            rescheduling_cost_per_change=reschedule_cost,
            scenarios=problem.scenarios,
        )

        sensitivity_model, sensitivity_x, sensitivity_y, sensitivity_z = (
            solve_stochastic_programming(modified_problem)
        )
        scenario_assignments, scenario_final_assignments, scenario_waiting_times = (
            extract_solution(
                modified_problem,
                sensitivity_x,
                sensitivity_y,
                sensitivity_z,
            )
        )

        reschedule_count = sum(
            scenario_final_assignments[s_idx][k] != scenario_assignments[k]
            for s_idx in range(len(modified_problem.scenarios))
            for k in range(1, modified_problem.num_trucks + 1)
        )

        results.append(
            {
                "Rescheduling Cost": reschedule_cost,
                "Total Expected Cost": pulp.value(sensitivity_model.objective),
                "Number of Reschedules": reschedule_count,
            }
        )

    return pd.DataFrame(results)


print("Performing sensitivity analysis on rescheduling cost...")
sensitivity_df = sensitivity_analysis(problem)
print(sensitivity_df.round(2).to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(
    sensitivity_df["Rescheduling Cost"],
    sensitivity_df["Total Expected Cost"],
    "o-",
    linewidth=2,
    markersize=8,
    color="#1f77b4",
)
ax1.set_xlabel("Rescheduling cost per change ($)")
ax1.set_ylabel("Total expected cost ($)")
ax1.set_title("Expected Cost versus Rescheduling Cost")
ax1.grid(True, alpha=0.3)

ax2.plot(
    sensitivity_df["Rescheduling Cost"],
    sensitivity_df["Number of Reschedules"],
    "o-",
    linewidth=2,
    markersize=8,
    color="#ff7f0e",
)
ax2.set_xlabel("Rescheduling cost per change ($)")
ax2.set_ylabel("Number of truck-scenario changes")
ax2.set_title("Rescheduling Decisions versus Rescheduling Cost")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Results and interpretation

The printed assignments and cost table are the results for the supplied scenario data and cost parameters. Read the initial assignments as decisions made before the delay scenario is known. Read each scenario's final assignments as recourse decisions made after that scenario is observed.

The scenario-cost table reports operating, waiting, and rescheduling components separately. Its expected-cost contribution column multiplies each scenario's total cost by that scenario's probability; summing these contributions gives the model's objective value. The assertion in the cost-analysis cell checks this accounting identity against the solver objective.

The sensitivity table and plots show how the optimal expected cost and the number of truck-scenario changes vary over the tested rescheduling costs. A higher rescheduling cost makes changes more expensive, but the exact assignment pattern can depend on the full set of costs and capacity constraints. If multiple assignments have the same optimal objective, the solver may return any one of them; the sensitivity count can therefore reflect the selected optimum in a tied case.

### Practical interpretation and limits

- Scenario probabilities express how strongly each modeled operating condition contributes to expected cost.
- Slot capacity restricts both the initial plan and every scenario-specific final plan.
- The sensitivity analysis illustrates the modeled value of flexibility under different rescheduling costs; it does not change the main example's parameters.
- The model treats waiting as a delay-based cost for reassigned trucks and does not calculate queue formation, physical arrival feasibility, or interactions between trucks. Those operational features would require additional model data and constraints.

The result is an optimal solution to the stated two-stage stochastic mixed-integer model, not a claim that the model includes every operational feature of a real terminal.